# Morphological Profiling

Evaluate segmentation models on the **U2OS-Cell-Painting dataset** — a 5-channel Cell Painting screen of human U2OS osteosarcoma cells treated with 231 compounds. The pipeline benchmarks models on their ability to preserve morphological signal, measured by how well unsupervised clustering separates Mechanism of Action (MoA) classes.

**Dataset:** 231 compounds at a single 10 µM dose (+ DMSO controls), 10 balanced MoA classes, 18 plates × 5 sites = 7,710 fluorescence fields (2160×2160, uint16), 5 channels (DNA, Mito, AGP, RNA, ER).

**References:**
- Data source: figshare record `21378906`, DOI [10.17044/scilifelab.21378906](https://doi.org/10.17044/scilifelab.21378906) (Uppsala University, CC BY 4.0)
- Gupta, Harrison, *et al.* "Is brightfield all you need for mechanism of action prediction?" *bioRxiv* 2022. DOI [10.1101/2022.10.12.511869](https://doi.org/10.1101/2022.10.12.511869)
- Tian, Harrison, *et al.* "Combining molecular and cell painting image data for mechanism of action prediction." *Artificial Intelligence in the Life Sciences*, 2023. DOI [10.1016/j.ailsci.2023.100060](https://doi.org/10.1016/j.ailsci.2023.100060)

**Pipeline:**
1. Preprocess metadata
2. Whole-cell segmentation (AGP membrane + DNA nucleus)
3. Feature extraction (~169 features per cell × 5 channels)
4. Aggregation & normalization
5. Unsupervised clustering & evaluation


In [ ]:
import sys
from pathlib import Path

# Setup paths
SRC_DIR = Path.cwd().parent / 'src'
MORPH_DIR = SRC_DIR / 'morphology_profiling'
if str(MORPH_DIR) not in sys.path:
    sys.path.insert(0, str(MORPH_DIR))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import config
config.ensure_dirs()

print(f'Project root: {config.PROJECT_ROOT}')
print(f'Data root: {config.DATA_ROOT}')
print(f'Results dir: {config.RESULTS_DIR}')


## Download Data

Download metadata and plate images from [figshare 21378906](https://doi.org/10.17044/scilifelab.21378906). The full record is ~634 GB (each plate tar.gz bundles FL + brightfield; only the 5 FL channels are extracted). For full-scale runs prefer `run_pipeline.py`, which processes plate by plate and deletes images after featurization (~200 GB peak disk).


In [ ]:
# Download metadata only (label table fl_data.csv) - run this first
# !python {MORPH_DIR / 'download.py'} --metadata_only

# Download a single plate for testing (~4.6 GB for the smallest plate)
# !python {MORPH_DIR / 'download.py'} --plate P015080

# Download all 18 plates (~634 GB; prefer run_pipeline.py --all instead)
# !python {MORPH_DIR / 'download.py'} --all

# Check what data is available
if config.METADATA_DIR.exists():
    csvs = list(config.METADATA_DIR.glob('*.csv'))
    print(f'Metadata files: {[f.name for f in csvs]}')
if config.IMAGES_DIR.exists():
    plates = [d.name for d in config.IMAGES_DIR.iterdir() if d.is_dir()]
    print(f'Downloaded plates: {len(plates)} ({plates[:5]}...)')
else:
    print('No image data found. Run download commands above.')


## Step 1: Preprocess Metadata

Build the unified image table from `fl_data.csv` (plate/well/site, compound, per-channel TIFF filenames and the single MoA label per field — no external platemap join needed).


In [ ]:
from preprocess import run_preprocessing

df, summary_df = run_preprocessing()

print(f'\nImage table: {len(df):,} rows')
print(f'Treatment summary: {len(summary_df)} treatments')
print(f'\nMoA classes: {df[df["moa"] != "unknown"]["moa"].nunique()}')
print(f'DMSO fields: {df["is_dmso"].sum()}')


## Step 2: Segmentation

Run whole-cell instance segmentation for all fields: cytoplasm/membrane marker **AGP** paired with the **DNA** nucleus marker (single AGP marker for microsam).

**Models:** cellpose4, cellpose3, microatlas, microsam, cellsam


In [ ]:
from segment import load_model, segment_plate

# Select model
MODEL_NAME = 'microatlas'  # Options: 'cellpose4', 'cellpose3', 'microatlas', 'microsam', 'cellsam'

# Get available plates
plates = [d.name for d in sorted(config.IMAGES_DIR.iterdir())
          if d.is_dir() and (d / config.NUCLEUS_CHANNEL).exists()]
print(f'Available plates: {len(plates)}')
print(f'Model: {MODEL_NAME}')

# Load model
model = load_model(MODEL_NAME, use_gpu=True)
config.ensure_model_dirs(MODEL_NAME)

# Segment all plates
for plate in plates:
    print(f'\n--- Plate: {plate} ---')
    segment_plate(plate, MODEL_NAME, model)


## Step 3: Feature Extraction

Extract ~169 morphological features per cell across 6 categories (5 channels):

| Category | Description |
|---|---|
| AreaShape | regionprops shape descriptors |
| Intensity | batched ndimage stats + percentiles × 5ch |
| Texture | Haralick GLCM on DNA + RNA |
| Granularity | multi-scale opening on DNA |
| RadialDistribution | binned radial intensity × 5ch |
| Correlation | Pearson + Manders inter-channel |


In [ ]:
from feature_extraction import extract_plate_features

# Extract features for all plates
for plate in plates:
    print(f'\n--- Extracting features: {plate} ---')
    extract_plate_features(plate, MODEL_NAME)

# Check output
feat_dir = config.features_dir(MODEL_NAME)
feat_files = list(feat_dir.glob('features_*.csv'))
print(f'\nFeature files generated: {len(feat_files)}')


## Step 4: Aggregation & Normalization

Aggregate single-cell features to field-level (median + MAD), robust z-score against per-plate DMSO controls, remove low-variance (< 0.01) and highly correlated (> 0.95) features.


In [ ]:
from feature_aggregation import run_aggregation

# Aggregate to field-level, normalize against DMSO, select features
df_field, df_treatment = run_aggregation(model_name=MODEL_NAME)
print(f'\nField-level profiles: {len(df_field)} fields')
meta_cols = {"plate", "field", "compound", "concentration", "moa",
             "is_dmso", "well", "n_cells"}
print(f'Features after selection: {len([c for c in df_field.columns if c not in meta_cols])}')


## Step 5: Unsupervised Clustering & Evaluation

Field-level profiles are reduced via PCA (50 dims) then UMAP (5D, cosine distance), clustered via **Agglomerative Clustering (ward linkage, k=10)**, and reduced to compound level via majority vote. Quality is measured by Hungarian-matched Accuracy — the proportion of compounds correctly assigned to their MoA class — with 95% bootstrap confidence intervals; NMI and ARI are reported alongside.


In [ ]:
from biomarker.unsupervised import run_field_level_clustering

# Run the field-level clustering pipeline (k=10, compound majority vote)
field_umap_df, cluster_df, enrichment_df, metrics = run_field_level_clustering(
    model_name=MODEL_NAME, mode="field")

print(f'\n{"="*60}')
print(f'Results for {MODEL_NAME}:')
print(f'  Hungarian Accuracy: {metrics["Accuracy"]:.4f} '
      f'[{metrics.get("Accuracy_CI_low", float("nan")):.4f}, '
      f'{metrics.get("Accuracy_CI_high", float("nan")):.4f}]')
print(f'  NMI: {metrics["NMI"]:.4f}')
print(f'  ARI: {metrics["ARI"]:.4f}')
print(f'{"="*60}')
